In [8]:
pip install sage.all
pip install sage

SyntaxError: invalid syntax (2122806141.py, line 1)

In [2]:
# For n > 2d, directed distances are 0, 1, or 2.
def razdalja(w, u, n, d):
    if n <= 2 * d:
        raise ValueError("The distance shortcut requires n > 2d.")
    if w == u:
        return 0
    korak = (u - w) % n
    return 1 if 1 <= korak <= n - d else 2

In [7]:
from sage.all import *

def metricna_dimenzija_Cn(n, d, prelomi_simetrijo=True):
    """
    Izračuna metrično dimenzijo usmerjenega cirkulantnega grafa C_n(1, ..., n-d).
    Predpostavka: n > 2*d (zato so razdalje le 0, 1 ali 2).
    """
    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen!")
    
    # 2. Nastavitev linearnega programa (ILP)
    lp = MixedIntegerLinearProgram(maximization=False)
    x = lp.new_variable(binary=True)
    
    # Ciljna funkcija: minimiziramo število izbranih radarjev
    lp.set_objective(sum(x[w] for w in range(n)))
    
    # Lomljenje rotacijske simetrije (priporočeno v navodilih za hitrost pri večjih n)
    if prelomi_simetrijo:
        lp.add_constraint(x[0] == 1)

    # 3. Pogoji: vsak par (u, v) mora biti ločen z vsaj enim radarjem
    for u in range(n):
        for v in range(u + 1, n):  # Gledamo samo u < v (brez podvajanja!)
            # Poiščemo vsa vozlišča w, ki ločijo par u in v
            locevalci = [w for w in range(n) if razdalja(w, u, n, d) != razdalja(w, v, n, d)]
            
            # Dodamo pogoj: vsaj eden izmed ločevalcev mora biti izbran
            lp.add_constraint(sum(x[w] for w in locevalci) >= 1)
            
    # 4. Rešitev modela
    lp.solve()
    
    # Pridobimo izbrana vozlišča (pazimo na zaokroževanje pri float vrednostih)
    razresujoca_mnozica = [w for w in range(n) if round(lp.get_values(x[w])) == 1]
    dimenzija = len(razresujoca_mnozica)
    
    return dimenzija, razresujoca_mnozica

ModuleNotFoundError: No module named 'sage'

In [ ]:
def je_razresujoca_mnozica(n, d, W):
    """Independently verify that W distinguishes all vertices."""
    if len(W) != len(set(W)) or any(w < 0 or w >= n for w in W):
        return False
    podpisi = [tuple(razdalja(w, v, n, d) for w in W) for v in range(n)]
    return len(set(podpisi)) == n


def test_znanih_formul(n_max=40):
    """Test the ILP against the known exact formulas for d=4 and d=5."""
    if n_max < 11:
        raise ValueError("n_max must be at least 11 to test both formulas.")

    stevilo_testov = 0
    for d, n_min in ((4, 10), (5, 11)):
        for n in range(n_min, n_max + 1):
            dimenzija, W = metricna_dimenzija_Cn(n, d)

            if len(W) != dimenzija:
                raise AssertionError(f"Returned dimension and set size disagree for n={n}, d={d}.")
            if not je_razresujoca_mnozica(n, d, W):
                raise AssertionError(f"Returned set does not resolve the graph for n={n}, d={d}.")

            if d == 4:
                expected = (2 * n + 4) // 5
            elif n % 3 == 0:
                expected = n // 3
            else:
                expected = n // 3 + 2

            if dimenzija != expected:
                raise AssertionError(
                    f"Known formula mismatch for n={n}, d={d}: got {dimenzija}, expected {expected}."
                )
            stevilo_testov += 1

    print(f"All {stevilo_testov} known-formula tests passed.")


# Run this cell after the solver-definition cell.
test_znanih_formul(n_max=40)